## **Dataset Creation & Preprocessing Notebook Summary**
This notebook outlines the process for combining, cleaning, and standardizing multiple public datasets to create a unified corpus for emotion classification from text. The goal is to prepare a high-quality, large-scale dataset featuring seven target emotions: **neutral, anger, disgust, fear, happiness, sadness, and surprise.**

### Core Components and Tools
- **Libraries**: `pandas`, Hugging Face `datasets`.

- **Target Schema**: All input data is mapped to a standard set of 7 emotions based on the original `roskoN/dailydialog` emotion mapping.

- **Preprocessing Functions**: Utility functions are defined to flatten nested datasets (like DailyDialog) and to apply standardized cleaning (typo correction, name unification, and filtering to the 7 target emotions).

### **Datasets Integrated**
The notebook processes and integrates three distinct datasets:

1. **Dataset 1 (Kaggle CSV)**: A sentiment and emotion analysis dataset focused primarily on **joy, sadness, anger, fear, and surprise.** The cleaning step converts labels like "joy" to "happiness" and filters out non-target labels like "love."

2. **Dataset 2 (roskoN/dailydialog)**: A dialogue-based dataset that includes the **neutral** label. The data, which is structured as lists of sentences and lists of corresponding emotion IDs, is flattened and the numerical IDs are mapped to string emotion names.

3. **Dataset 3 (boltuix/emotions-dataset)**: A large dataset with 13 distinct labels. The cleaning step standardizes existing labels and filters out non-target emotions such as "confusion," "shame," and "guilt."

### **Merging and Final Output**

1. **Concatenation**: The three cleaned DataFrames (`df1_cleaned`, `df2_cleaned`, `df3_cleaned`) are concatenated into a single large DataFrame (`df_concat`).

2. **ID Mapping**: A final column, `Emotion_ID`, is added by mapping the standardized emotion names back to their original numerical IDs (0-6).

3. **Deduplication**: Duplicate sentences are identified and removed to prevent data leakage and ensure model robustness. Over **64,000 duplicate entries** were dropped.

The final dataset is ready for feature extraction and model training, containing a consolidated, cleaned, and standardized set of sentences and their corresponding emotion labels.



In [1]:
import pandas as pd
from datasets import DatasetDict, load_dataset
from typing import Dict, Any, List, Union
from functools import partial

c:\Users\rzvn1\.conda\envs\nlp_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Emotion Map obtained from https://aclanthology.org/I17-1099/ --> readme.txt
EMOTION_MAP: Dict[int, str] = {
    0: "neutral", 1: "anger", 2: "disgust", 3: "fear",
    4: "happiness", 5: "sadness", 6: "surprise"
}

# The final set of target emotions derived from the original code's filtering
TARGET_EMOTIONS = {
    "neutral", "anger", "disgust", "fear",
    "happiness", "sadness", "surprise"
}

# Standardized emotion name replacements
EMOTION_REPLACEMENTS = {
    'joy': 'happiness',
    'sad': 'sadness',
    'suprise': 'surprise'
}

In [3]:
def dataset_to_dataframe(
    dataset: DatasetDict,
    sentence_col: str,
    emotion_col: str,
    nested: bool = False
) -> pd.DataFrame:
    """
    Flattens a Hugging Face DatasetDict into a single pandas DataFrame
    where each row is a sentence-emotion pair.

    Args:
        dataset: The Hugging Face DatasetDict to process.
        sentence_col: The name of the column containing the sentence/utterance.
        emotion_col: The name of the column containing the emotion label/ID.
        nested: If True, assumes a nested structure (e.g., lists of utterances/emotions).

    Returns:
        A pandas DataFrame with 'Sentence', 'Emotion_Raw', and 'Split' columns.
    """
    all_data: List[Dict[str, Any]] = []

    for split_name, ds_split in dataset.items():
        if nested:
            # Handle nested structure (e.g., DailyDialog)
            for example in ds_split:
                for sentence, emotion in zip(example[sentence_col], example[emotion_col]):
                    all_data.append({
                        'Sentence': sentence,
                        'Emotion_Raw': emotion,
                        'Split': split_name
                    })
        else:
            # Handle flat structure (e.g., emotions-dataset)
            for example in ds_split:
                all_data.append({
                    'Sentence': example[sentence_col],
                    'Emotion_Raw': example[emotion_col],
                    'Split': split_name
                })

    return pd.DataFrame(all_data)

In [4]:
def clean_dataframe(
    df: pd.DataFrame,
    id_to_emotion_map: Union[Dict[Any, str], None] = None,
    target_emotions: set = TARGET_EMOTIONS,
    replacements: Dict[str, str] = EMOTION_REPLACEMENTS
) -> pd.DataFrame:
    """
    Applies standard cleaning and filtering steps to a DataFrame.

    - Renames 'Emotion_Raw' to 'Emotion' or maps IDs to names.
    - Unifies emotion names (e.g., 'joy' -> 'happiness').
    - Filters to keep only the target emotions.
    """
    if id_to_emotion_map:
        # Map IDs to string names
        df['Emotion'] = df['Emotion_Raw'].map(id_to_emotion_map)
    else:
        # Rename the raw column directly for string-based emotions
        df.rename(columns={'Emotion_Raw': 'Emotion'}, inplace=True)

    # Standardize emotion names (e.g., joy -> happiness, suprise -> surprise)
    df['Emotion'] = df['Emotion'].replace(replacements)

    # Filter out emotions not in the final target list
    mask_to_keep = df['Emotion'].isin(target_emotions)
    df = df[mask_to_keep].reset_index(drop=True)

    # Return only the core columns
    return df[['Sentence', 'Emotion']]

## Loading the datasets

### Dataset 1 - Sentiment and Emotion Analysis Dataset

The dataset can be found at [https://www.kaggle.com/datasets/kushagra3204/sentiment-and-emotion-analysis-dataset?resource=download](https://www.kaggle.com/datasets/kushagra3204/sentiment-and-emotion-analysis-dataset?resource=download)

The dataset contains over 422,000 sentences, labeled with six distinct emotions:

- Joy: 143,067 samples
- Sadness: 121,187 samples
- Anger: 59,317 samples
- Fear: 49,649 samples
- Love: 34,554 samples
- Surprise: 14,972 samples


In [5]:
# https://www.kaggle.com/datasets/kushagra3204/sentiment-and-emotion-analysis-dataset?resource=download
# Sentiment and Emotion Analysis Dataset

df1 = pd.read_csv(r"..\Data\CSV\sentiment_data\combined_emotion.csv")
df1['emotion'].value_counts()

emotion
joy        143067
sad        121187
anger       59317
fear        49649
love        34554
suprise     14972
Name: count, dtype: int64

### Dataset 2 - roskoN/dailydialog

The dataset can be found at [https://huggingface.co/datasets/roskoN/dailydialog](https://huggingface.co/datasets/roskoN/dailydialog)

The dataset contains 102,979 sentences, labeled with seven distinct emotions, including neutral:

- Neutral: 85572
- Happiness: 12885
- Surprise: 1823
- Hadness: 1150
- Anger: 1022
- Disgust: 353
- Fear: 174


In [6]:
ds2 = load_dataset("roskoN/dailydialog")

### Dataset 3 - boltuix/emotions-dataset

The dataset can be found at [https://huggingface.co/datasets/boltuix/emotions-dataset](https://huggingface.co/datasets/boltuix/emotions-dataset)

The dataset contains 131,306 sentences, labeled with 13 distinct emotions:
- 😊 Happiness: 31,205 (23.76%)
- 😢 Sadness: 17,809 (13.56%)
- 😐 Neutral: 15,733 (11.98%)
- 😣 Anger: 13,341 (10.16%)
- ❤️ Love: 10,512 (8.00%)
- 😨 Fear: 8,795 (6.70%)
- 🤢 Disgust: 8,407 (6.40%)
- ❓ Confusion: 8,209 (6.25%)
- 😲 Surprise: 4,560 (3.47%)
- 😳 Shame: 4,248 (3.24%)
- 😔 Guilt: 3,470 (2.64%)
- 😏 Sarcasm: 2,534 (1.93%)
- 💫 Desire: 2,483 (1.89%)

In [7]:
ds3 = load_dataset("boltuix/emotions-dataset")

## Preprocessing

### Dataset 1 (Kaggle CSV)

The CSV-loaded dataset (df1) is processed using the unified clean_dataframe function after initial loading and column renaming.

1. **Column Alignment**: The columns are renamed from `sentence` and `emotion` to the standardized `Sentence` and `Emotion_Raw` upon loading to prepare for cleaning.

2. **Name Standardization & Typo Correction**: The `clean_dataframe` function applies the `EMOTION_REPLACEMENTS` map to standardize names and correct typos in a single step: 

    - "joy" → "happiness"

    - "sad" → "sadness"

    - "suprise" → "surprise"

3. **Emotion Filtering (Implicit Removal)**: The dataset is filtered to only keep emotions present in the global `TARGET_EMOTIONS` set. This effectively removes labels like `"love"` and others that are not part of the final seven-emotion schema.

In [8]:
# 1. Process df1 (CSV-loaded)
df1 = pd.read_csv(r"..\Data\CSV\sentiment_data\combined_emotion.csv")

In [9]:
# Correct column names for mapping
df1.rename(columns={
    'sentence': 'Sentence',
    'emotion': 'Emotion_Raw'
}, inplace=True)

In [10]:
df1_cleaned = clean_dataframe(df1, id_to_emotion_map=None) # Clean and standardize the dataframe

In [11]:
df1_cleaned['Emotion'].value_counts()

Emotion
happiness    143067
sadness      121187
anger         59317
fear          49649
surprise      14972
Name: count, dtype: int64

### Dataset 2 (roskoN/dailydialog) 

This dataset is handled using the general `dataset_to_dataframe` function with the `nested=True` flag, followed by the `clean_dataframe` function.

1. **Dataframe Conversion & Unpacking**: The `dataset_to_dataframe function` iterates through the DatasetDict. Since the data has a nested dialogue structure (each example contains lists of sentences and emotions), the `nested=True` flag ensures that each dialogue is unpacked to create a single row for every individual sentence-emotion pair.

2. **Emotion Mapping**: The raw emotion column, `Emotion_Raw`, which contains numerical IDs, is mapped to string names using the `EMOTION_MAP` constant (derived from the original research paper's `readme.txt`). This mapping is passed into clean_dataframe via `functools.partial`.

3. **Consolidated Cleaning**: The dataset is implicitly cleaned and filtered as part of the `clean_dataframe` execution, which includes name standardization and filtering down to the `TARGET_EMOTIONS`.

In [12]:
# 2. Process ds2 (roskoN/dailydialog)
ds2 = load_dataset("roskoN/dailydialog")   

In [13]:
df2 = dataset_to_dataframe(
    ds2,
    sentence_col='utterances',
    emotion_col='emotions',
    nested=True
)  

In [14]:
# Use functools.partial to pre-fill the map argument for cleaner execution
clean_df2_with_map = partial(clean_dataframe, id_to_emotion_map=EMOTION_MAP)
df2_cleaned = clean_df2_with_map(df2)  

In [15]:
df2_cleaned['Emotion'].value_counts()

Emotion
neutral      85572
happiness    12885
surprise      1823
sadness       1150
anger         1022
disgust        353
fear           174
Name: count, dtype: int64

### Dataset 3 (boltuix/emotions-dataset)

This Hugging Face dataset is processed using the general `dataset_to_dataframe` function with the default `nested=False` flag, followed by `the clean_dataframe` function.

1. **Dataframe Conversion (Flat Structure)**: The `dataset_to_dataframe function` loads the dataset, which has a flat structure (one sentence and one label per example). The columns `Sentence` and `Label` are mapped to the standardized `Sentence` and `Emotion_Raw`.

2. **Emotion Filtering**: This dataset contains a wider variety of emotion labels (e.g., 'confusion', 'shame', 'guilt', 'sarcasm', 'desire'). The `clean_dataframe` function automatically filters out all labels that are not present in the global `TARGET_EMOTIONS` set. This ensures only the final seven target emotions remain for merging.

3. **Consolidated Cleaning**: Since the labels are already strings, no ID mapping is applied. The data is simply filtered and checked against the standard `EMOTION_REPLACEMENTS` (though this dataset generally doesn't require them).

In [16]:
# 3. Process ds3 (boltuix/emotions-dataset)
ds3 = load_dataset("boltuix/emotions-dataset")

In [17]:
df3 = dataset_to_dataframe(
    ds3,
    sentence_col='Sentence',
    emotion_col='Label',
    nested=False
)

In [18]:
df3_cleaned = clean_dataframe(df3, id_to_emotion_map=None)

In [19]:
df3_cleaned['Emotion'].value_counts()

Emotion
happiness    31205
sadness      17809
neutral      15733
anger        13341
fear          8795
disgust       8407
surprise      4560
Name: count, dtype: int64

## Merging the Datasets

In [20]:
df_concat = pd.concat(
    [df1_cleaned, df2_cleaned, df3_cleaned],
    ignore_index=True
)

In [21]:
reverse_map = {v: k for k, v in EMOTION_MAP.items()}

In [22]:
df_concat['Emotion_ID'] = df_concat['Emotion'].map(reverse_map)
df_concat['Emotion_ID'].astype(int)
print()

In [23]:
df_concat.head()

,Sentence,Emotion,Emotion_ID
0,i just feel really helpless and heavy hearted,fear,3
1,ive enjoyed being able to slouch about relax a...,sadness,5
2,i gave up my internship with the dmrg and am f...,fear,3
3,i dont know i feel so lost,sadness,5
4,i am a kindergarten teacher and i am thoroughl...,fear,3


In [24]:
df_concat['Sentence'].duplicated().sum()

64726

In [25]:
df_concat = df_concat.drop_duplicates(subset=['Sentence'], keep='first')

In [26]:
df_concat['Emotion'].value_counts()

Emotion
happiness    172941
sadness      130590
neutral       84981
anger         64930
fear          49574
surprise      15708
disgust        7571
Name: count, dtype: int64

In [27]:
output_path = r"..\Data\CSV\sentiment_data\final_dataset.csv"
df_concat.to_csv(output_path)